In [2]:
import pandas as pd
import yfinance as yf
import requests
from google.cloud import bigquery
from google.oauth2 import service_account
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

start_date = '2018-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')

print("⏳ Đang cào dữ liệu Cổ phiếu, Vàng, FX bằng yfinance...")
tickers = ['GC=F', 'EURUSD=X', 'HPG.VN', 'VNM.VN', 'TCB.VN']
df_market = yf.download(tickers, start=start_date, end=end_date, progress=False)['Close']
df_market.index = pd.to_datetime(df_market.index).tz_localize(None).normalize()

df_market.rename(columns={
    'GC=F': 'VANG', 
    'EURUSD=X': 'EURUSD',
    'HPG.VN': 'HPG',
    'VNM.VN': 'VNM',
    'TCB.VN': 'TCB'
}, inplace=True)
print("⏳ Đang hút dữ liệu Phái sinh VN30F1M từ Server DNSE...")
# Chuyển ngày sang định dạng Timestamp mà Server yêu cầu
start_ts = int(pd.Timestamp(start_date).timestamp())
end_ts = int(pd.Timestamp(end_date).timestamp())

# Gửi HTTP Request thẳng vào API lõi
url = f"https://services.entrade.com.vn/chart-api/v2/ohlcs/derivative?from={start_ts}&to={end_ts}&symbol=VN30F1M&resolution=1D"
response = requests.get(url).json()

# Biến JSON thành DataFrame
dates_utc = pd.to_datetime(response['t'], unit='s', utc=True)
dates_vn = dates_utc.tz_convert('Asia/Ho_Chi_Minh').tz_localize(None).normalize()
df_ps = pd.DataFrame({'VN30F1M': response['c']}, index=dates_vn)
# ==============================================================
# GỘP TOÀN BỘ VÀ LÀM SẠCH
# ==============================================================
print("-> Đang gộp danh mục Đa tài sản và xử lý Ngày nghỉ lễ...")
# JOIN OUTER: Ghép toàn bộ, không bỏ sót ngày nào của cả 2 thị trường
df_master = df_market.join(df_ps, how='outer')

# KỸ THUẬT FORWARD FILL: Đắp giá hôm trước vào những ngày nghỉ lễ
df_master.ffill(inplace=True)

# Chỉ dropna lần cuối để xóa những ngày đầu tiên (khi có 1 tài sản chưa niêm yết)
df_master.dropna(inplace=True)

# Chuẩn bị mặt tiền cho BigQuery
df_master.reset_index(inplace=True)
if 'index' in df_master.columns:
    df_master.rename(columns={'index': 'Date'}, inplace=True)
df_master['Date'] = pd.to_datetime(df_master['Date']).astype(str)

print(f"\n✅ Xem trước dữ liệu (Tổng số: {len(df_master)} dòng):")
print(df_master.tail())

# SETUP BIGQUERY VÀ ĐẨY DATA
project_id = "jda-k1"
dataset_id = "cuongnm_data_pipeline"
table_id = "Market_Risk_Data"
key_path = "C:/Users/DELL/Desktop/Project/Project Market risk/Key - path 2afad6d8f47e.json"

credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=project_id)

table_ref = f"{project_id}.{dataset_id}.{table_id}"

# Lần khởi tạo đầu tiên dùng TRUNCATE. 
# (Sau này ráp vào pipeline chạy hàng ngày thì sửa chữ này thành APPEND)
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    autodetect=True,  
)

#Load DataFrame vào BigQuery
job = client.load_table_from_dataframe(df_master, table_ref, job_config=job_config)
job.result()  

print(f"✅ Đã tải {job.output_rows} dòng vào bảng {table_ref}")

print("\nXem trước dữ liệu:")
print(df_master.head())

⏳ Đang cào dữ liệu Cổ phiếu, Vàng, FX bằng yfinance...
⏳ Đang hút dữ liệu Phái sinh VN30F1M từ Server DNSE...
-> Đang gộp danh mục Đa tài sản và xử lý Ngày nghỉ lễ...

✅ Xem trước dữ liệu SIÊU SẠCH (Tổng số: 2082 dòng):
            Date    EURUSD         VANG      HPG      TCB      VNM  VN30F1M
2077  2026-07-13  1.140446  3997.000000  22400.0  32350.0  55400.0   1941.5
2078  2026-07-14  1.138433  4061.100098  22500.0  32200.0  56000.0   1943.5
2079  2026-07-15  1.142465  4044.000000  22400.0  31500.0  56200.0   1919.1
2080  2026-07-16  1.147026  3985.600098  22200.0  31900.0  56200.0   1936.3
2081  2026-07-17  1.144558  4012.699951  21850.0  31450.0  59000.0   1930.0
✅ Đã tải 2082 dòng vào bảng jda-k1.cuongnm_data_pipeline.Market_Risk_Data

Xem trước dữ liệu:
         Date    EURUSD         VANG          HPG           TCB           VNM  \
0  2018-08-13  1.139471  1191.300049  9002.343750  25795.814453  71860.679688   
1  2018-08-14  1.140251  1193.000000  9049.353516  25560.878906  727